# Stochastic Calculus, Step by Step · Lesson 17: Geometric Brownian motion

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The model · 3. Solving it with d ln S · 4. Back to Lesson 5: the lognormal · 5. E[S²] by Itô · 6. Euler's method against the exact solution · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026
mu = 0.03   # the share's expected return; equal to r here, so results match Lesson 5

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 17,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. The model

**Geometric Brownian motion**: $dS_t = \mu S_t\,dt + \sigma S_t\,dW_t$. Over a short step the share's **return** $dS/S$ has mean $\mu\,dt$ and standard deviation $\sigma\sqrt{dt}$, whatever the price level. We take $\mu = r = 3\%$, so every result can be checked against Lesson 5.

In [ ]:
rng = np.random.default_rng(seed)
n_paths, n_steps = 50_000, 252
dt = T / n_steps
dW = rng.normal(0, math.sqrt(dt), (n_paths, n_steps))
W = np.cumsum(dW, axis=1)
grid = np.linspace(dt, T, n_steps)
S = S0 * np.exp((mu - 0.5 * sigma**2) * grid + sigma * W)            # exact solution on the grid (Section 3)
out["model"] = {"mu": mu, "fan": {"t": [0.0] + grid[3::4].tolist(),
                                   "paths": [[S0] + S[k_, 3::4].round(3).tolist() for k_ in range(10)],
                                   "ticks": [{"x": x, "label": f"{x:g}"} for x in (0, 0.25, 0.5, 0.75, 1)]}}
ret = dW[:, 0] * sigma + mu * dt
out["model"]["one_step"] = {"mean": float(ret.mean()), "mean_theory": mu * dt, "sd": float(ret.std()),
                            "sd_theory": sigma * math.sqrt(dt)}
tf = np.array(out["model"]["fan"]["t"])
out["model"]["fan"]["mean"] = (S0 * np.exp(mu * tf)).tolist()
out["model"]["fan"]["median"] = (S0 * np.exp((mu - 0.5 * sigma**2) * tf)).tolist()
out["model"]["one_step"]

## 3. Solving it with d ln S

Apply Itô's lemma (Lesson 16) to $f(S) = \ln S$, with $f' = 1/S$ and $f'' = -1/S^2$:

$$d\ln S = \frac{1}{S}dS - \frac{1}{2S^2}(dS)^2 = \mu\,dt + \sigma\,dW - \tfrac12\sigma^2\,dt .$$

So $\ln S_t = \ln S_0 + (\mu - \tfrac12\sigma^2)t + \sigma W_t$, and $S_t = S_0e^{(\mu - \frac12\sigma^2)t + \sigma W_t}$. The **log** grows at $\mu - \tfrac12\sigma^2$, not $\mu$.

In [ ]:
logret = np.log(S[:, -1] / S0)
out["log"] = {"mean": float(logret.mean()), "mean_theory": (mu - 0.5 * sigma**2) * T, "var": float(logret.var()),
              "var_theory": sigma**2 * T, "half_sigma2": 0.5 * sigma**2}
out["log"]

## 4. Back to Lesson 5: the lognormal

$\ln S_T$ is normal, so $S_T$ is lognormal, exactly as in Lesson 5. Its mean is $S_0e^{\mu T}$, its median $S_0e^{(\mu - \frac12\sigma^2)T}$, its mode $S_0e^{(\mu - \frac32\sigma^2)T}$; and the chance of ending above its own mean is $N(-\tfrac12\sigma\sqrt T)$, less than a half.

In [ ]:
ST = S[:, -1]
mean_th = S0 * math.exp(mu * T); med_th = S0 * math.exp((mu - 0.5 * sigma**2) * T)
lognormal = {"mean": float(ST.mean()), "mean_theory": mean_th, "median": float(np.median(ST)), "median_theory": med_th,
             "mode_theory": S0 * math.exp((mu - 1.5 * sigma**2) * T),
             "p_above_mean": float((ST > mean_th).mean()), "p_above_mean_theory": N(-0.5 * sigma * math.sqrt(T)),
             "p_below_S0": float((ST < S0).mean()), "p_below_S0_theory": N(-(mu - 0.5 * sigma**2) * T / (sigma * math.sqrt(T)))}
out["lognormal"] = lognormal
lognormal

## 5. E[S²] by Itô

$d(S^2) = 2S\,dS + (dS)^2 = (2\mu + \sigma^2)S^2\,dt + 2\sigma S^2\,dW$. Averaging, $m(t) = E[S_t^2]$ solves $m' = (2\mu + \sigma^2)m$ (Lesson 2), so $E[S_T^2] = S_0^2e^{(2\mu + \sigma^2)T}$ and $\mathrm{Var}(S_T) = S_0^2e^{2\mu T}(e^{\sigma^2T} - 1)$.

In [ ]:
ES2 = S0**2 * math.exp((2 * mu + sigma**2) * T)
out["second"] = {"ES2": float((ST**2).mean()), "ES2_theory": ES2, "var_theory": ES2 - mean_th**2,
                 "sd_theory": math.sqrt(ES2 - mean_th**2), "sd": float(ST.std()), "rate": 2 * mu + sigma**2}
out["second"]

## 6. Euler's method against the exact solution

**Euler's method** (Lesson 2) steps the equation directly: $S_{k+1} = S_k(1 + \mu\Delta t + \sigma\Delta W_k)$. Driven by the same $\Delta W$ as the exact solution, its path-by-path (**strong**) error shrinks like $\sqrt{\Delta t}$, and its error in the mean (**weak**) like $\Delta t$. Dropping the $-\tfrac12\sigma^2$ from the exact formula is a different matter: that error never shrinks.

In [ ]:
paths = 20_000
euler_rows = []
fine_steps = 1024
dWf = rng.normal(0, math.sqrt(T / fine_steps), (paths, fine_steps))
exact = S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * dWf.sum(axis=1))
wrong = S0 * np.exp(mu * T + sigma * dWf.sum(axis=1))                  # the -sigma^2/2 deleted
for n in (4, 16, 64, 256, 1024):
    dWn = dWf.reshape(paths, n, fine_steps // n).sum(axis=2)           # the same noise on a coarser grid
    eul = S0 * np.prod(1 + mu * (T / n) + sigma * dWn, axis=1)
    euler_rows.append({"n": n, "strong": float(np.abs(eul - exact).mean()), "weak": abs(S0 * (1 + mu * T / n) ** n - mean_th),
                       "wrong": float(np.abs(eul - wrong).mean())})
out["euler"] = {"rows": euler_rows, "wrong_mean": float(wrong.mean()),
                "wrong_mean_theory": S0 * math.exp((mu + 0.5 * sigma**2) * T),
                "strong_ratio": euler_rows[-2]["strong"] / euler_rows[-1]["strong"],
                "ticks": [{"x": x, "label": f"{x:g}"} for x in (4, 16, 64, 256, 1024)]}
pd.DataFrame(euler_rows)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 3: Solutions*.

**Exercise 1.** Derive $d\ln S_t$ for $dS = \mu S\,dt + \sigma S\,dW$ with Itô's lemma, step by step, and solve for $S_t$.

**Exercise 2.** Check that $S_t = S_0e^{(\mu - \frac12\sigma^2)t + \sigma W_t}$ satisfies the equation, by applying the two-variable Itô's lemma to $f(t, w) = S_0e^{(\mu - \frac12\sigma^2)t + \sigma w}$.

**Exercise 3.** With $\mu = 3\%$, $\sigma = 20\%$, $T = 1$, find the mean, median and mode of $S_T$, and $P(S_T > E[S_T])$. Why is the last one below a half?

**Exercise 4.** Use Itô's lemma on $S^2$ to find $E[S_T^2]$ and $\mathrm{Var}(S_T)$. Compare with Lesson 5's formula.

**Exercise 5.** Show that $e^{-\mu t}S_t$ is a martingale. What does this say when $\mu = r$?

**Exercise 6.** Find the distribution of $\ln(S_T/S_0)$ and compute $P(S_1 < 100)$ and $P(S_1 < 80)$.

**Exercise 7.** (Code) Compare Euler's method with the exact solution, driven by the same noise, for $n = 4, 16, \dots, 1024$ steps: the average size of the error at $T$ (strong) and the error in the mean (weak). Estimate how each shrinks.

**Exercise 8.** (Code) The **Milstein** scheme adds one term: $S_{k+1} = S_k\big(1 + \mu\Delta t + \sigma\Delta W_k + \tfrac12\sigma^2(\Delta W_k^2 - \Delta t)\big)$. Repeat the strong-error test and compare with Euler.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")